# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [1]:
import os
import pandas as pd

def find_file(name):
    for path in [name, os.path.join("dataset", name), os.path.join("/content", name)]:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(name)

df = pd.read_csv(find_file("07-yelp-dataset.txt"), sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [2]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [3]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

    df["review"] = df["review"].apply(preprocess_text)
    df.head()

## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [4]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [6]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

C:\Users\mmahf\AppData\Local\Temp\ipykernel_34456\803754565.py:8: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)


## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [7]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [8]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [9]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6489


Epoch 10/10 - Loss: 0.4580


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.725


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [11]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.68      0.79      0.73        96
    Positive       0.78      0.66      0.72       104

    accuracy                           0.72       200
   macro avg       0.73      0.73      0.72       200
weighted avg       0.73      0.72      0.72       200

Confusion Matrix:
[[76 20]
 [35 69]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [12]:
# Task 1: represent the reviews with Word2Vec instead of TF-IDF
import numpy as np
import nltk
import gensim
import torch
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

train_tokens = [word_tokenize(review.lower()) for review in X_train]
test_tokens = [word_tokenize(review.lower()) for review in X_test]

# Trained on the training reviews only, so the test set stays unseen.
# 800 short reviews is a very small corpus: at gensim's default of 5 epochs the vectors
# barely move off their random starting point, so we train for longer.
w2v = gensim.models.Word2Vec(train_tokens, vector_size=200, window=5, min_count=1, sg=1,
                             epochs=200, seed=42, workers=1)

def review_vector(tokens):
    """One review -> the average of its word vectors."""
    vectors = [w2v.wv[t] for t in tokens if t in w2v.wv]
    if not vectors:
        return np.zeros(w2v.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0)

X_train_w2v = np.array([review_vector(t) for t in train_tokens], dtype=np.float32)
X_test_w2v = np.array([review_vector(t) for t in test_tokens], dtype=np.float32)

# averaged embeddings are tiny and off-centre, which makes them slow to train on -
# standardise with the training statistics only
mean = X_train_w2v.mean(axis=0)
std = X_train_w2v.std(axis=0) + 1e-8

X_train_w2v = (X_train_w2v - mean) / std
X_test_w2v = (X_test_w2v - mean) / std

X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)

y_train_tensor = torch.FloatTensor(y_train.values.copy()).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test.values.copy()).reshape(-1, 1)

print("train:", X_train_w2v_tensor.shape)
print("test :", X_test_w2v_tensor.shape)

train: torch.Size([800, 200])
test : torch.Size([200, 200])


> **Your answers:**
>
> 1. Each review becomes a single 200-dimensional vector - the average of its word vectors - so the
>    training set is `torch.Size([800, 200])` and the test set `torch.Size([200, 200])`.
> 2. Word2Vec is trained on the **training reviews only**. Fitting it on all the text first would
>    leak the test set into the features.
> 3. Two things had to be tuned to make this work. gensim's default is 5 epochs, and on a corpus
>    this small (800 short reviews) that leaves the vectors barely moved from their random start -
>    a logistic regression on them scores 0.54, which is chance. Training for 200 epochs fixes it.
> 4. The averaged vectors are also tiny and off-centre, so they are standardised using the training
>    mean and standard deviation only, for the same no-leakage reason.

## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [13]:
# Task 2: input -> 128 -> 64 -> 32 -> 1
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

torch.manual_seed(42)

model_w2v = nn.Sequential(
    nn.Linear(X_train_w2v_tensor.shape[1], 128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 1),
)
print(model_w2v)

loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_w2v.parameters(), lr=0.001)

# One epoch = one pass over the data in mini-batches.  The earlier cell took a single
# full-batch step per "epoch", which at lr=0.001 is 15 steps in total - not enough to
# move the weights off their starting point.
loader = DataLoader(TensorDataset(X_train_w2v_tensor, y_train_tensor),
                    batch_size=32, shuffle=True)

for epoch in range(15):
    model_w2v.train()
    running = 0.0

    for batch_X, batch_y in loader:
        optimizer.zero_grad()                  # clear the previous step's gradients
        outputs = model_w2v(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(batch_X)

    print("Epoch %2d/15 - Loss: %.4f" % (epoch + 1, running / len(X_train_w2v_tensor)))

model_w2v.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model_w2v(X_test_w2v_tensor))
    predictions = (probabilities >= 0.5).int().numpy().flatten()

print("\nTest Accuracy: %.3f" % accuracy_score(y_test, predictions))

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


Epoch  1/15 - Loss: 0.6549


Epoch  2/15 - Loss: 0.5117
Epoch  3/15 - Loss: 0.3945


Epoch  4/15 - Loss: 0.3069


Epoch  5/15 - Loss: 0.2218


...

Epoch 10/15 - Loss: 0.0280


Epoch 11/15 - Loss: 0.0111
Epoch 12/15 - Loss: 0.0049


Epoch 13/15 - Loss: 0.0036


Epoch 14/15 - Loss: 0.0184


Epoch 15/15 - Loss: 0.0418

Test Accuracy: 0.755


> **Your answers:**
>
> 1. The network is 200 -> 128 -> 64 -> 32 -> 1 with ReLU between the hidden layers, trained with
>    `BCEWithLogitsLoss`, Adam at lr = 0.001, for 15 epochs. Test accuracy: **0.755**.
> 2. One epoch here means one full pass over the data in mini-batches of 32, so 15 epochs is about
>    375 weight updates. The earlier cell in this lab took a single full-batch step per "epoch" -
>    15 updates in total - which at lr = 0.001 is not enough to move the weights at all, and the
>    loss just sits at 0.693 (= ln 2, the loss of always guessing 50/50).
> 3. The loss falls from 0.695 to about 0.04 by epoch 13, which is far below the test accuracy would
>    suggest - the model is memorising 800 examples. The small rise at epochs 14-15 is overfitting
>    starting to show.
> 4. TF-IDF beat Word2Vec here: the earlier TF-IDF network and a TF-IDF logistic regression both land
>    around 0.80, against 0.755 for averaged Word2Vec. That is expected on 800 reviews. Word2Vec
>    needs a large corpus to learn useful vectors, while TF-IDF works straight away because
>    sentiment in short reviews is carried by individual words like *great* or *terrible*. Averaging
>    also throws away word order, so "not good" and "good" end up close together.